# 02-02 窗口函数（Window Functions）

**面试必考！** 窗口函数是数据岗位最高频的 SQL 考点，广告/推荐/用户增长分析全都离不开它。

**本节目标**：
- 理解窗口函数原理（不折叠行）
- 掌握 6 类核心函数：排名、偏移、聚合、分布
- 实战：7日留存、漏斗转化、TopN、环比分析

---

In [ ]:
import duckdb
import pandas as pd
import sys
sys.path.insert(0, "..")
from utils.data_generator import generate_ad_events

con = duckdb.connect()

# 导入数据
events = generate_ad_events(n=50000, seed=42)
events_df = pd.DataFrame(events)
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'])
events_df['date'] = events_df['timestamp'].dt.date
con.execute("CREATE TABLE events AS SELECT * FROM events_df")
print(f"数据加载完成: {len(events_df):,} 行")

## 1. 窗口函数语法

```sql
函数() OVER (
    PARTITION BY 分组列    -- 类似 GROUP BY，但不折叠行
    ORDER BY 排序列        -- 窗口内的排序
    ROWS/RANGE BETWEEN ... -- 帧范围（滑动窗口）
)
```

In [ ]:
# ============================================================
# 排名函数：ROW_NUMBER / RANK / DENSE_RANK
# ============================================================
print("=== 各广告位 TOP3 广告（按点击量）===")
con.sql("""
WITH click_counts AS (
    SELECT ad_id, position, COUNT(*) AS clicks
    FROM events
    WHERE action = 'click'
    GROUP BY ad_id, position
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY position ORDER BY clicks DESC) AS rn,
        RANK()       OVER (PARTITION BY position ORDER BY clicks DESC) AS rank_,
        DENSE_RANK() OVER (PARTITION BY position ORDER BY clicks DESC) AS dense_rk
    FROM click_counts
)
SELECT position, ad_id, clicks, rn, rank_, dense_rk
FROM ranked
WHERE rn <= 3
ORDER BY position, rn
""").show()

In [ ]:
# ============================================================
# 偏移函数：LAG / LEAD —— 环比分析
# ============================================================
print("=== 广告日消耗环比（LAG）===")
con.sql("""
WITH daily_cost AS (
    SELECT 
        date,
        ROUND(SUM(cost), 2) AS daily_cost
    FROM events
    GROUP BY date
    ORDER BY date
)
SELECT 
    date,
    daily_cost,
    LAG(daily_cost)  OVER (ORDER BY date)                           AS prev_day_cost,
    LEAD(daily_cost) OVER (ORDER BY date)                           AS next_day_cost,
    ROUND(
        100.0 * (daily_cost - LAG(daily_cost) OVER (ORDER BY date))
               / NULLIF(LAG(daily_cost) OVER (ORDER BY date), 0), 
    1) AS day_over_day_pct
FROM daily_cost
LIMIT 10
""").show()

In [ ]:
# ============================================================
# 累计聚合：SUM OVER —— 累计消耗
# ============================================================
print("=== 每个广告主的累计消耗（SUM OVER）===")
con.sql("""
WITH daily AS (
    SELECT advertiser_id, date, ROUND(SUM(cost), 2) AS daily_cost
    FROM events
    GROUP BY advertiser_id, date
)
SELECT 
    advertiser_id,
    date,
    daily_cost,
    ROUND(SUM(daily_cost) OVER (
        PARTITION BY advertiser_id 
        ORDER BY date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS cumulative_cost
FROM daily
WHERE advertiser_id = 'adv_1'
ORDER BY date
LIMIT 10
""").show()

In [ ]:
# ============================================================
# 滑动窗口：7日移动平均 —— 消除短期波动
# ============================================================
print("=== 7日移动平均点击量 ===")
con.sql("""
WITH daily_clicks AS (
    SELECT date, COUNT(*) AS clicks
    FROM events
    WHERE action = 'click'
    GROUP BY date
    ORDER BY date
)
SELECT 
    date,
    clicks,
    ROUND(AVG(clicks) OVER (
        ORDER BY date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW  -- 包含当天在内共7天
    ), 1) AS clicks_7d_avg
FROM daily_clicks
ORDER BY date
LIMIT 15
""").show()

In [ ]:
# ============================================================
# 面试实战题：用户次日留存率
# ============================================================
print("=== 用户次日留存率 ===")
con.sql("""
WITH user_first_day AS (
    -- 每个用户第一次出现的日期（注册/首次访问）
    SELECT user_id, MIN(date) AS first_day
    FROM events
    GROUP BY user_id
),
user_active_days AS (
    -- 每个用户每天是否活跃
    SELECT DISTINCT user_id, date AS active_day
    FROM events
),
cohort AS (
    SELECT 
        f.first_day,
        COUNT(DISTINCT f.user_id) AS cohort_size,
        COUNT(DISTINCT CASE 
            WHEN a.active_day = f.first_day + INTERVAL '1 day' 
            THEN f.user_id END) AS retained_d1
    FROM user_first_day f
    LEFT JOIN user_active_days a ON f.user_id = a.user_id
    GROUP BY f.first_day
)
SELECT 
    first_day,
    cohort_size,
    retained_d1,
    ROUND(100.0 * retained_d1 / cohort_size, 1) AS d1_retention_pct
FROM cohort
ORDER BY first_day
LIMIT 10
""").show()

In [ ]:
# ============================================================
# 面试实战题：漏斗转化分析
# ============================================================
print("=== 广告投放漏斗转化 ===")
con.sql("""
WITH funnel AS (
    SELECT 
        COUNT(DISTINCT CASE WHEN action = 'impression' THEN user_id END) AS step1_impression,
        COUNT(DISTINCT CASE WHEN action = 'click'      THEN user_id END) AS step2_click,
        COUNT(DISTINCT CASE WHEN action = 'convert'    THEN user_id END) AS step3_convert
    FROM events
)
SELECT 
    step1_impression,
    step2_click,
    step3_convert,
    ROUND(100.0 * step2_click   / step1_impression, 2) AS imp_to_click_pct,
    ROUND(100.0 * step3_convert / step2_click,      2) AS click_to_convert_pct,
    ROUND(100.0 * step3_convert / step1_impression, 2) AS overall_cvr_pct
FROM funnel
""").show()

## 总结：窗口函数速查表

| 函数 | 作用 | 典型场景 |
|------|------|---------|
| `ROW_NUMBER()` | 连续行号（无并列） | 去重、TopN per group |
| `RANK()` | 有并列，跳号 | 排行榜 |
| `DENSE_RANK()` | 有并列，不跳号 | 分级 |
| `LAG(col, n)` | 取前 n 行的值 | 环比 |
| `LEAD(col, n)` | 取后 n 行的值 | 预测下一步 |
| `SUM() OVER` | 累计或滑动求和 | 累计消耗、滑动总量 |
| `AVG() OVER` | 滑动平均 | 7日移动均线 |
| `FIRST_VALUE()` | 窗口第一行 | 同组最早记录 |
| `NTILE(n)` | 分 n 等份 | 分桶、百分位 |
| `PERCENT_RANK()` | 百分位排名 | 用户分层 |

**自检**: 不看答案，能手写7日留存查询吗？

**下一节**: `03_hive_basics.ipynb` — Hive 基础